# 9.2. Converting Raw Text into Sequence Data
D2L의 Converting Raw Text into Sequence Data장을 PyTorch 기준으로 정리함.

RNN이나 Trnasformer에 문장을 그대로 넣을 수는 없는데 예를 들어서

    the time machine

이라는 문자열이 있다고 해보자. 신경망은 문자열 자체를 계산할 수 없으므로 최종적으로 다음과 같은 숫자 형태로 변환해야 한다.

    the time machine -> ['t', 'h', 'e', ' ', 't', ...] -> [21, 9, 6, 0, 21, ...]

일반적인 텍스트 전처리 과정은 다음과 같다.

    Raw Text -> 전처리 -> Tokenization -> Vocabulary 생성 -> 숫자 index로 변환 -> Sequence Data

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

## 1. 필요한 라이브러리

이번장은 H. G. Wells의 소설 The Time Machine을 사용한다.

중요한 건 `텍스트 -> 토큰 -> 숫자` 로 변환하는 전체과정을 이해하는 것이다.

In [ ]:
import collections 
import re 
import torch 

from d2l import torch as d2l

## 2. 텍스트 데이터 읽기

먼저 텍스트 파일을 하나의 긴 문자열로 읽는다.

In [ ]:
class TimeMachine(d2l.DataModule):
    def _download(self):
        fname = d2l.download(
            d2l.DATA_URL + 'timemachine.txt',
            self.root,
            '090b5e7e70c295757f55df93cb0a180b9691891a'
        )

        with open(fname) as f:
            return f.read()

In [ ]:
data = TimeMachine() 

raw_text = data._download() 

print(raw_text[:100])

현재 상태에선 대문자, 쉼표, 마침표 등 여러 문자가 섞여 있다. 학습하기 전 텍스트를 정리한다.

## 3. 텍스트 전처리

이번 예제에서는 단순화를 위해서 `대문자 -> 소문자`, `알파벳이 아닌 문자 -> 공백` 으로 변경한다.

In [ ]:
def preprocess(text):
    text = re.sub('[^A-Za-z]+', ' ', text)

    return text.lower()
text = preprocess(raw_text)

print(text[:100])

주의할 점은 실제 NLP에선 항상 문장부호를 제거하지 않는다. 여기에서는 쉽게 설명하기 위해 단순한 전처리를 한다.

## 4. Tokenization

이제 문자열을 토큰(token) 으로 나눈다. 토큰은 모델이 처리하는 텍스트의 기본 단위이다.

대표적으로 `문자(character)`, `단어(word)`, `subword`를 사용할 수 있다.

예를 들어서 deep learning을 나누면 이렇게 된다.

    단어 단위로 나누면 ['deep', 'learning'] 
    문자 단위로 나누면 ['d', 'e', 'e', 'p', ' ', 'l', ...]

이번 장에서는 문자 단위 토큰화를 사용한다.

`Sequence의 한 시점(time step) = 하나의 Token`

문자 단위 모델이라면 t -> h -> e -> ... 각 문자가 하나의 time step이다.

In [ ]:
def tokenize(text):
    return list(text)
tokens = tokenize(text)

print(tokens[:30])

## 5. Vocabulary가 필요한 이유

현재 토큰은 여전히 문자열이다. ['t', 'h', 'e']

하지만 신경망은 문자열을 직접 계산하지 못한다. 그래서 각 토큰에 정수 번호를 부여한다.

예를 들어서 

```text
' ' → 0
'a' → 1
'b' → 2
'e' → 5
'h' → 8
't' → 20
```

이런 대응 관계 표를 만든다. 이 대응표를 Vocabulary(어휘집) 이라고 한다.

Token <-> Index의 관계를 저장하고 있는 것이다.

예를 들어서 [`t', 'h', 'e'] -> [20, 8, 5]로 된다.

중요한건 숫자 크기에 의미가 있는 건 아니다. 단순한 `ID번호`다.

## 6. Vocabulary 구현

In [ ]:
class Vocab:

    def __init__(self, tokens, min_freq=0):

        counter = collections.Counter(tokens)

        self.token_freqs = sorted(
            counter.items(),
            key=lambda x: x[1],
            reverse=True
        )

        unique_tokens = [
            token
            for token, freq in self.token_freqs
            if freq >= min_freq
        ]

        self.idx_to_token = ['<unk>'] + sorted(set(unique_tokens))

        self.token_to_idx = {
            token: idx
            for idx, token in enumerate(self.idx_to_token)
        }

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, token):
        return self.token_to_idx.get(
            token,
            self.token_to_idx['<unk>']
        )

<unk>는 unknown token을 의미한다. Vocabulary에 존재하지 않는 토큰이 입력되면 <unk>로 처리한다.

예를 들어 학습 데이터에 zebra라는 단어가 전혀 없었다면 새로운 데이터의 zebra를 <unk>로 처리할 수 있다.

## 7. Token을 숫자 Sequence로 변환하기

Vocabulary를 만들어보자.

처음 몇 개 문자를 숫자로 변환하면 정수 시퀀스로 바뀐다.

    ['t', 'h', 'e', ' ', 't', ...] -> [21, 9, 6, 1, 21, ...]

이것이 RNN에서 사용할 수 있는 데이터의 기본 형태이다.

전체 흐름을 다시 보면 이렇다.

```text
"the time machine"

        ↓ Tokenization

['t', 'h', 'e', ' ', ...]

        ↓ Vocabulary

t → 21
h → 9
e → 6
  → 0

        ↓

[21, 9, 6, 0, ...]
```

앞으로 RNN이 실제로 받는 것은 문자열이 아니라 이런 token index sequence이다.

In [ ]:
vocab = Vocab(tokens)

indices = [
    vocab[token]
    for token in tokens[:10]
]

print(indices)


## 8. Corpus 만들기

지금까지의 과정을 하나로 합칠 수 있다.

아래에서 corpus는 전체 텍스트를 숫자로 바꾼 하나의 긴 sequence이다.

예를 들어서 the time 이라는 텍스트가 있으면 [21, 9, 6, 1, 21, 10, 14, 6] 같은 형태가 된다.

모델은 `Token ID Sequence`를 보게 된다.

In [ ]:
def build_corpus(raw_text):

    text = preprocess(raw_text)

    tokens = tokenize(text)

    vocab = Vocab(tokens)

    corpus = [
        vocab[token]
        for token in tokens
    ]

    return corpus, vocab
corpus, vocab = build_corpus(raw_text)

print(len(corpus))
print(len(vocab))

## 9. 단어 빈도와 Zipf's Law

이번에는 문자 대신 단어 단위로 텍스트를 살펴보자. 영어 문서에서는 일반적으로

```text
the 
a 
of 
to 
in
```
같은 단어가 매우 많이 등장한다. 이런 단어들은 문법적으로 중요하지만 특정 문서의 내용을 설명하는 정보는 상대적으로 적을 수 있다.

과거 Bag-of-words 기반 NLP에서는 이런 단어를 stop word라고 부르며 제거하는 경우가 많았다. 하지만 RNN이나 Transformer 같은 모델에서는 단어 사이의 문맥 자체를 학습하므로 반드시 제거할 필요는 없다.

단어 빈도를 순위별로 조하사면 매우 흥미로운 규칙이 나타난다. 이것을 `Zipf's Law`라고 한다.

$$
n_i \propto \frac{1}{i^\alpha}
$$

$i$ : 단어의 빈도 순위
$n_i$ : 해당 단어의 등장 횟수
$\alpha$ : 분포를 결정하는 값

로그를 취하면 이렇게 된다.
$$
\log n_i = -\alpha \log i + c
$$

따라서 단어 순위와 빈도를 log-log 그래프로 그리면 대략 직선 형태가 나타난다.

    일부 단어 -> 엄청 많이 등장
    대부분의 단어 -> 매우 적게 등장

NLP 데이터가 매우 불균형한 분포를 가진다는 뜻이다.

In [ ]:
words = text.split()

word_vocab = Vocab(words)
word_vocab.token_freqs[:10]

## 10. Unigram, Bigram, Trigram

지금까지는 각각의 단어 하나만 살펴봤다. 단어 하나를 unigram이라고 한다. 

예를 들어서 `I love deep learning`의 unigram은

```text
I 
love 
deep 
learning
```

이렇다. 두 단어씩 묶으면 bigram이다.

```text
I love
love deep
deep learning
```

세 단어씩 묶으면 trigram이다.

```text
I love deep 
love deep learning
```

일반적으로 연속된 $n$개의 토큰을 묶은 것을 n-gram이라고 한다. n-gram을 분석해도 Zipf's Law와 비슷한 빈도 패턴이 나타난다. 하지만 $n$이 커질수록 가능한 조합이 급격하게 많아지고 대부분의 조합은 매우 적게 등장한다.

예를 들어 Vocabulary 크기가 $V$라면 단순 계산상 가능한 bigram은 $V^2$개이고 trigram은 $V^3$개가 될 수 있다.

그래서 모든 문장 패턴을 직접 세어서 기억하는 방식은 한계가 있다. 이러한 문제 때문에 단순한 빈도 통계보다 문맥과 패턴 자체를 학습하는 신경망 기반 언어 모델이 필요해진다.

In [ ]:
bigrams = [
    (words[i], words[i + 1])
    for i in range(len(words) - 1)
]

trigrams = [
    (words[i], words[i + 1], words[i + 2])
    for i in range(len(words) - 2)
]

## 11. 오늘의 정리

- 신경망은 문자열을 직접 처리할 수 없기 때문에 텍스트를 숫자 sequence로 변환해야 한다.
- 일반적인 흐름은 Text → Token → Vocabulary → Index Sequence이다.
- Token은 모델이 처리하는 텍스트의 기본 단위이며 문자, 단어, subword 등을 사용할 수 있다.
- Vocabulary는 각 token과 정수 index 사이의 대응 관계를 저장한다.
- Vocabulary에 없는 token은 보통 <unk> 같은 특수 token으로 처리한다.
- Corpus는 전체 텍스트를 token index로 변환한 데이터이다.
- RNN에서는 이 token들이 시간 순서대로 입력되는 sequence가 된다.
- 자연어의 단어 빈도는 일부 단어가 매우 많이 등장하고 대부분은 드물게 등장하는 Zipf 분포를 보인다.
- 연속된 1개, 2개, 3개의 token을 각각 unigram, bigram, trigram이라고 부른다.
- n-gram이 길어질수록 가능한 조합은 많아지고 실제 관측 횟수는 희소해진다.
- 다음 장의 Language Model에서는 이 sequence를 이용해 앞의 token들을 보고 다음 token의 확률을 예측하는 방법을 다룬다.